In [1]:
# 1. Install standard client and UI tracking bars
!pip install chembl_webresource_client tqdm pandas

# 2. Install RDKit (Crucial biotech tool)
!pip install rdkit

# 3. Import everything to make sure it worked perfectly
from chembl_webresource_client.new_client import new_client
import pandas as pd
from tqdm import tqdm
import rdkit

print("All biotech libraries are successfully configured!")


   ---------------------------------------- 0.0/24.6 MB ? eta -:--:--
    --------------------------------------- 0.5/24.6 MB 5.6 MB/s eta 0:00:05
   -- ------------------------------------- 1.8/24.6 MB 6.3 MB/s eta 0:00:04
   ---- ----------------------------------- 2.9/24.6 MB 6.0 MB/s eta 0:00:04
   ------- -------------------------------- 4.5/24.6 MB 6.5 MB/s eta 0:00:04
   ---------- ----------------------------- 6.3/24.6 MB 7.0 MB/s eta 0:00:03
   ------------ --------------------------- 7.6/24.6 MB 7.2 MB/s eta 0:00:03
   -------------- ------------------------- 8.9/24.6 MB 6.9 MB/s eta 0:00:03
   ----------------- ---------------------- 10.7/24.6 MB 7.1 MB/s eta 0:00:02
   ------------------- -------------------- 11.8/24.6 MB 6.9 MB/s eta 0:00:02
   --------------------- ------------------ 13.1/24.6 MB 6.8 MB/s eta 0:00:02
   ----------------------- ---------------- 14.4/24.6 MB 6.8 MB/s eta 0:00:02
   -------------------------- ------------- 16.3/24.6 MB 6.9 MB/s eta 0:00:02
 

C:\Users\ajitc\anaconda3\Lib\site-packages\chembl_webresource_client\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


All biotech libraries are successfully configured!


In [2]:
# Search ChEMBL for the lung cancer target protein
target = new_client.target
target_query = target.search('EGFR')
targets_df = pd.DataFrame.from_dict(target_query)

# Filter out only human results (Homo sapiens)
human_targets = targets_df[targets_df['organism'] == 'Homo sapiens']

# Print the top rows to find our distinct ID
human_targets[['target_chembl_id', 'pref_name', 'target_type']].head(5)


,target_chembl_id,pref_name,target_type
0,CHEMBL4523747,EGFR/PPP1CA,PROTEIN-PROTEIN INTERACTION
1,CHEMBL5465557,CCN2-EGFR,PROTEIN-PROTEIN INTERACTION
4,CHEMBL203,Epidermal growth factor receptor,SINGLE PROTEIN
5,CHEMBL4523680,Protein cereblon/Epidermal growth factor receptor,PROTEIN-PROTEIN INTERACTION
6,CHEMBL2363049,Epidermal growth factor receptor,PROTEIN FAMILY


In [3]:
# 1. Specify our lung cancer protein family target ID
target_id = 'CHEMBL2363049'

# 2. Query the live database for bioactivities tied to this target
activities = new_client.activity
res = activities.filter(target_chembl_id=target_id).filter(standard_type="IC50")

# 3. Convert the server responses into a structured data table
print("Fetching experimental data from ChEMBL server...")
df = pd.DataFrame.from_dict(res)

# 4. View how much data we retrieved
print(f"Success! Retrieved a total of {len(df)} molecule bioactivity measurements.")
df[['molecule_chembl_id', 'canonical_smiles', 'standard_value', 'standard_units']].head(5)


Fetching experimental data from ChEMBL server...
Success! Retrieved a total of 95 molecule bioactivity measurements.


,molecule_chembl_id,canonical_smiles,standard_value,standard_units
0,CHEMBL225519,O=C1NCCc2[nH]c(-c3ccncc3)cc21,10000.0,nM
1,CHEMBL1256436,Cc1cc(Nc2nc3cccc(Cl)c3o2)ccc1-c1nn([C@H]2CC[C@...,530.0,nM
2,CHEMBL1256435,CN1CCN([C@H]2CC[C@H](n3nc(-c4ccc(Nc5nc6cccc(Cl...,86.0,nM
3,CHEMBL1256434,Cc1cc(-c2nn([C@H]3CC[C@H](N4CCN(C)CC4)CC3)c3nc...,156.0,nM
4,CHEMBL1256433,CN1CCN([C@H]2CC[C@H](n3nc(-c4ccc(Nc5nc6cccc(Cl...,63.0,nM


In [4]:
# 1. Drop rows that are missing a chemical structure (SMILES) or an experimental measurement
df_clean = df.dropna(subset=['canonical_smiles', 'standard_value'])

# 2. Convert the experimental concentration values to numbers so Python can analyze them
df_clean['standard_value'] = pd.to_numeric(df_clean['standard_value'])

# 3. View the clean biological dataframe
print(f"Cleaned Data: {len(df_clean)} molecules are fully complete and ready for AI feature extraction.")
df_clean[['molecule_chembl_id', 'canonical_smiles', 'standard_value']].head(5)


Cleaned Data: 94 molecules are fully complete and ready for AI feature extraction.


C:\Users\ajitc\AppData\Local\Temp\ipykernel_4748\4097207422.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['standard_value'] = pd.to_numeric(df_clean['standard_value'])


,molecule_chembl_id,canonical_smiles,standard_value
0,CHEMBL225519,O=C1NCCc2[nH]c(-c3ccncc3)cc21,10000.0
1,CHEMBL1256436,Cc1cc(Nc2nc3cccc(Cl)c3o2)ccc1-c1nn([C@H]2CC[C@...,530.0
2,CHEMBL1256435,CN1CCN([C@H]2CC[C@H](n3nc(-c4ccc(Nc5nc6cccc(Cl...,86.0
3,CHEMBL1256434,Cc1cc(-c2nn([C@H]3CC[C@H](N4CCN(C)CC4)CC3)c3nc...,156.0
4,CHEMBL1256433,CN1CCN([C@H]2CC[C@H](n3nc(-c4ccc(Nc5nc6cccc(Cl...,63.0


In [5]:
# Create a proper copy to cleanly get rid of that pink warning box
df_final = df_clean.copy()

# Define the labeling rule based on drug concentration threshold
def assign_activity(ic50_value):
    if ic50_value <= 1000:
        return 1  # Potent/Active compound
    else:
        return 0  # Inactive compound

# Run the rule across our dataset to create our AI target column
df_final['activity_label'] = df_final['standard_value'].apply(assign_activity)

# Count how many active and inactive molecules we have
print(df_final['activity_label'].value_counts())
print("\nMolecules successfully categorized!")


activity_label
1    74
0    20
Name: count, dtype: int64

Molecules successfully categorized!


In [6]:
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# 1. Function to convert a text SMILES string into a 2048-bit structural chemical array
def smiles_to_fingerprint(smiles_string):
    mol = Chem.MolFromSmiles(smiles_string)
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
        return np.array(fp)
    return None

# 2. Extract features (X) and labels (y)
print("Extracting chemical fingerprints using RDKit...")
X = np.array([smiles_to_fingerprint(s) for s in df_final['canonical_smiles']])
y = df_final['activity_label'].values

# 3. Split the data into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Initialize and Train the Random Forest Machine Learning Model
print("Training the Machine Learning Model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Test the model's predictions on unseen molecules
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n================ MODEL EVALUATION ================")
print(f"Machine Learning Model Accuracy Score: {accuracy * 100:.2f}%")
print("==================================================")


Extracting chemical fingerprints using RDKit...
Training the Machine Learning Model...


[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerator
[21:57:35] DEPRECATION WARNING: please use MorganGenerat


================ MODEL EVALUATION ================
Machine Learning Model Accuracy Score: 84.21%
